# S5 — Shade index

Stage 5 of the Manhattan Sidewalk Shade Index pipeline.

Aggregates the 11 per-timestep `shadows_HHMM.parquet` files (from S4) into `shade_index` and `peak_heat_index` per analysis unit. Per `docs/DECISIONS.md`, this join + aggregation runs in **DuckDB**.

**Accept when:** all indices ∈ [0, 1]; top-ranked streets are plausible on inspection; 5-unit visual spot-check documented in `docs/VALIDATION.md`.

In [1]:
from pathlib import Path
from datetime import datetime

import yaml
import duckdb
import pandas as pd
import geopandas as gpd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
config = yaml.safe_load(open(PROJECT_ROOT / "config.yaml"))
INTERIM = PROJECT_ROOT / config["interim_dir"]
ANALYSIS_CRS = config["analysis_crs"]
WINDOW_HOURS = config["end_hour"] - config["start_hour"]
PEAK_START, PEAK_END = config["peak_heat_window_start_hour"], config["peak_heat_window_end_hour"]
print("s5: Compute shade index")
print(f"Timestamp: {datetime.now().isoformat()}\n")

s5: Compute shade index
Timestamp: 2026-08-28T14:13:37.909991



## Load all timesteps, join to unit area, compute per-hour shade fraction

In [2]:
shadow_files = sorted(INTERIM.glob("shadows_*.parquet"))
print(f"found {len(shadow_files)} timestep files: {[f.stem for f in shadow_files]}")
assert shadow_files, "no shadows_HHMM.parquet files found -- run S4 first"

units = gpd.read_parquet(PROJECT_ROOT / config["output"]["analysis_units"])
units_scratch = INTERIM / "_scratch_units_s5.parquet"
units[["unit_id", "street_name", "side", "area_m2", "geometry"]].to_parquet(units_scratch)

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")
con.execute(f"CREATE TABLE units AS SELECT * EXCLUDE (geometry), geometry AS geom FROM read_parquet('{units_scratch.as_posix()}')")

hourly_frames = []
for f in shadow_files:
    hour_label = f.stem.replace("shadows_", "")
    con.execute(f"CREATE OR REPLACE TABLE shaded_{hour_label} AS SELECT * FROM read_parquet('{f.as_posix()}')")
    hourly_frames.append(hour_label)
print("loaded hours:", hourly_frames)

found 11 timestep files: ['shadows_0800', 'shadows_0900', 'shadows_1000', 'shadows_1100', 'shadows_1200', 'shadows_1300', 'shadows_1400', 'shadows_1500', 'shadows_1600', 'shadows_1700', 'shadows_1800']


loaded hours: ['0800', '0900', '1000', '1100', '1200', '1300', '1400', '1500', '1600', '1700', '1800']


In [3]:
# Long-format table: one row per (unit_id, hour), shade_fraction clipped to [0,1].
# LEFT JOIN so units with no shadow that hour get shaded_area_m2 = 0, not dropped.
union_parts = []
for hour_label in hourly_frames:
    union_parts.append(f"""
        SELECT u.unit_id, '{hour_label}' AS hour_label,
               LEAST(COALESCE(s.shaded_area_m2, 0) / u.area_m2, 1.0) AS shade_fraction
        FROM units u LEFT JOIN shaded_{hour_label} s ON u.unit_id = s.unit_id
    """)
con.execute("CREATE TABLE hourly_long AS " + " UNION ALL ".join(union_parts))
print(con.execute("SELECT count(*), min(shade_fraction), max(shade_fraction) FROM hourly_long").fetchone())

(380633, 0.0, 1.0)


## Aggregate: `shade_index`, `peak_heat_index`, hourly fractions as columns

In [4]:
peak_hours = [f"{h:02d}00" for h in range(PEAK_START, PEAK_END)]
print("peak heat window hours:", peak_hours)

summary = con.execute(f"""
    SELECT
        unit_id,
        SUM(shade_fraction) * {config['step_hours']} / {WINDOW_HOURS} AS shade_index,
        AVG(shade_fraction) FILTER (WHERE hour_label IN ({",".join(f"'{h}'" for h in peak_hours)})) AS peak_heat_index
    FROM hourly_long
    GROUP BY unit_id
""").fetchdf()

hourly_wide = con.execute("""
    PIVOT hourly_long ON hour_label USING SUM(shade_fraction) GROUP BY unit_id
""").fetchdf()
hourly_wide.columns = [c if c == "unit_id" else f"shade_fraction_{c}" for c in hourly_wide.columns]

units_attrs = con.execute("SELECT unit_id, street_name, side, area_m2 FROM units").fetchdf()
final = units_attrs.merge(summary, on="unit_id").merge(hourly_wide, on="unit_id")
final = final.merge(units[["unit_id", "geometry"]], on="unit_id")
final = gpd.GeoDataFrame(final, geometry="geometry", crs=ANALYSIS_CRS)
print(f"final: {len(final):,} rows, columns={list(final.columns)}")

peak heat window hours: ['1100', '1200', '1300', '1400', '1500']


final: 34,603 rows, columns=['unit_id', 'street_name', 'side', 'area_m2', 'shade_index', 'peak_heat_index', 'shade_fraction_0800', 'shade_fraction_0900', 'shade_fraction_1000', 'shade_fraction_1100', 'shade_fraction_1200', 'shade_fraction_1300', 'shade_fraction_1400', 'shade_fraction_1500', 'shade_fraction_1600', 'shade_fraction_1700', 'shade_fraction_1800', 'geometry']


## QA summary and validation

In [5]:
print("\n" + "=" * 70)
print("S5 — Shade index: QA Summary")
print("=" * 70)
print(final[["shade_index", "peak_heat_index"]].describe())
print("=" * 70)

assert final["shade_index"].between(0, 1).all(), "shade_index out of [0,1]"
assert final["peak_heat_index"].between(0, 1).all(), "peak_heat_index out of [0,1]"
assert final["unit_id"].is_unique
print("All S5 acceptance checks passed (indices in [0,1]).")

top20 = final.sort_values("peak_heat_index", ascending=False).head(20)
print("\nTop 20 shadiest Manhattan sidewalks by peak_heat_index:")
print(top20[["street_name", "side", "peak_heat_index", "shade_index"]].to_string(index=False))


S5 — Shade index: QA Summary
        shade_index  peak_heat_index
count  34603.000000     34603.000000
mean       0.084687         0.112612
std        0.110328         0.158552
min        0.000000         0.000000
25%        0.000000         0.000000
50%        0.034084         0.025120
75%        0.141709         0.185751
max        0.769175         0.968368
All S5 acceptance checks passed (indices in [0,1]).

Top 20 shadiest Manhattan sidewalks by peak_heat_index:
    street_name  side  peak_heat_index  shade_index
 WEST 81 STREET  Left         0.968368     0.693857
 WEST 81 STREET  Left         0.956579     0.731407
 EAST 13 STREET  Left         0.955318     0.525411
 EAST 97 STREET Right         0.947042     0.663934
 EAST 57 STREET  Left         0.946113     0.617832
 EAST 57 STREET  Left         0.942678     0.566116
DELANCEY STREET Right         0.903078     0.714020
 EAST 85 STREET Right         0.902919     0.513823
 WEST 77 STREET Right         0.901413     0.604022
       5

In [6]:
processed_dir = PROJECT_ROOT / config["processed_dir"]
processed_dir.mkdir(parents=True, exist_ok=True)

out_parquet = PROJECT_ROOT / config["output"]["shade_index_units"]
final.to_parquet(out_parquet)
print(f"Wrote {out_parquet}")

top20.drop(columns="geometry").to_csv(processed_dir / "top20_shadiest.csv", index=False)
print(f"Wrote {processed_dir / 'top20_shadiest.csv'}")

units_scratch.unlink()
print("\ns5 complete. Ready for S6 (tiles and web map).")

Wrote C:\Users\juanz\OneDrive\Desktop\Cursos\K3-MODERN-GIS\accelerator\accelerator\part4-cloud-capstone\4.4-coding-agents\data\processed\shade_index_units.parquet
Wrote C:\Users\juanz\OneDrive\Desktop\Cursos\K3-MODERN-GIS\accelerator\accelerator\part4-cloud-capstone\4.4-coding-agents\data\processed\top20_shadiest.csv

s5 complete. Ready for S6 (tiles and web map).
